# FLOW — K-12 Progressive Education Pipeline

**Geometric Causal Architecture** — weight-free, token-free reasoning.  
Knowledge stored as shape in a 104D Riemannian manifold.

This notebook builds a **progressive K-12 education pipeline** that grows the manifold
from Kindergarten vocabulary through Grade 12 advanced academics — each grade building
on the previous one, mirroring how students actually learn.

### Pipeline Design

| Stage | What Happens |
|---|---|
| Load Base Model | Load FLOW-V1-Base (703 concepts) from `models/` |
| Wire Grammar | Confirm Phase 10 Geometric Grammar Engine is active |
| K → Grade 12 | Progressive vocabulary + curriculum concept feeding |
| Contrast Pairs | SAME/DIFFERENT within each grade's domain |
| Grade-Level Queries | Ask questions, get grade-appropriate answers |
| Content Generation | Generate explanations at targeted reading levels |

### Grade Band Structure (Progressive)

```
K       → Sight words, colors, shapes, numbers, body, family, actions
1-2     → Fry words, basic science, math operations, community
3-5     → Academic vocab, ecosystems, fractions, geography, literary terms
6-8     → Hypothesis/evidence, biology, algebra, world history, rhetoric
9-12    → Advanced academic, thermodynamics, calculus, geopolitics, literary theory
```

**Data source:** 100% built-in word lists (Dolch, Fry, Common Core, academic vocab)  
**No external datasets required.**  
**Hardware:** Kaggle CPU (4 cores, 30 GB RAM)  
**Estimated runtime:** 10–20 minutes

## 1. Setup — Clone repo & install dependencies

In [ ]:
import os, subprocess, json

REPO_DIR = "FLOW"
REPO_URL = "https://github.com/Unseengap/FLOW"

# ── Read PAT from Kaggle Secrets (or environment) ─────────────────────
PAT = None
try:
    from kaggle_secrets import UserSecretsClient
    PAT = UserSecretsClient().get_secret("GITHUB_PAT")
    print("\u2713 PAT loaded from Kaggle Secrets")
except Exception:
    PAT = os.environ.get("GITHUB_PAT")
    if PAT:
        print("\u2713 PAT loaded from environment variable")
    else:
        print("\u2718 No PAT found \u2014 using public clone (read-only)")

clone_url = REPO_URL
if PAT:
    clone_url = f"https://{PAT}@github.com/Unseengap/FLOW"

# ── Clone or update ───────────────────────────────────────────────────
if os.path.isdir(REPO_DIR):
    print(f"\u2713 {REPO_DIR}/ already exists \u2014 pulling latest...")
    head_before = subprocess.check_output(
        ["git", "rev-parse", "--short", "HEAD"], cwd=REPO_DIR
    ).decode().strip()
    subprocess.run(["git", "fetch", "--all"], cwd=REPO_DIR, capture_output=True)
    subprocess.run(["git", "reset", "--hard", "origin/main"], cwd=REPO_DIR, capture_output=True)
    head_after = subprocess.check_output(
        ["git", "rev-parse", "--short", "HEAD"], cwd=REPO_DIR
    ).decode().strip()
    if head_before != head_after:
        print(f"  Updated: {head_before} \u2192 {head_after}")
    else:
        print(f"  Already up to date ({head_after})")
else:
    print(f"Cloning {REPO_URL} ...")
    subprocess.run(["git", "clone", clone_url, REPO_DIR], check=True)
    head = subprocess.check_output(
        ["git", "rev-parse", "--short", "HEAD"], cwd=REPO_DIR
    ).decode().strip()
    print(f"\u2713 Cloned at {head}")

# ── Install dependencies ──────────────────────────────────────────────
!pip install numpy scipy networkx -q

# Verify
assert os.path.isdir(f"{REPO_DIR}/src"), "FLOW repo not found"
print("\u2713 FLOW repository ready")

In [ ]:
# ── Add FLOW to Python path & import everything ──────────────────────
import sys
sys.path.insert(0, "FLOW")

import numpy as np
import time
from collections import Counter, OrderedDict

from src.phase5 import GEOPipeline, PipelineResult, PipelineEvaluator
from src.persistence import ManifoldSnapshot
from src.phase3.annealing_engine.experience import Experience
from src.phase1.expression.matcher import ExpressionEntry
from src.vocabulary import VocabularyStore, WordPlacer
from src.vocabulary.word_placer import structural_feature_vector

print(f"\u2713 All FLOW imports successful")
print(f"  numpy {np.__version__}")

# Check Phase 10 Grammar availability
try:
    from src.phase10 import GrammarRenderer, SyntaxGeometry, MorphologyMap
    print("\u2713 Phase 10 Grammar Engine available")
    GRAMMAR_AVAILABLE = True
except ImportError as e:
    print(f"\u2718 Phase 10 Grammar not available: {e}")
    GRAMMAR_AVAILABLE = False

## 2. Load Base Model & Confirm Grammar

In [ ]:
# ── Load FLOW-V1-Base ─────────────────────────────────────────────────
MANIFOLD_PATH = "FLOW/models/FLOW-V1-Base_manifold.npz"
VOCAB_PATH    = "FLOW/models/FLOW-V1-Base_vocab.npz"

if os.path.exists(MANIFOLD_PATH):
    print(f"Loading base model from {MANIFOLD_PATH} ...")
    pipeline = GEOPipeline.load(
        MANIFOLD_PATH,
        vocabulary_path=VOCAB_PATH,
        flow_seed=42,
    )
    print(f"\u2713 Base model loaded")
else:
    print("No saved model found \u2014 building fresh pipeline...")
    pipeline = GEOPipeline(T0=1.0, lambda_=0.005, T_floor=0.03, flow_seed=42)
    print(f"\u2713 Fresh pipeline built")

print(f"  Manifold dim       : {pipeline.manifold.DIM}")
print(f"  Concepts on M(t)   : {pipeline.n_concepts:,}")
print(f"  Temperature        : {pipeline.temperature:.6f}")
print(f"  Queries so far     : {pipeline.query_count}")

# ── Confirm grammar wiring ────────────────────────────────────────────
grammar_active = pipeline._renderer._grammar is not None
print(f"\n  Grammar engine     : {'\u2713 ACTIVE' if grammar_active else '\u2718 INACTIVE'}")
if grammar_active:
    g = pipeline._renderer._grammar
    print(f"    SyntaxGeometry   : {type(g.syntax).__name__}")
    print(f"    ClauseComposer   : {type(g.composer).__name__}")
    print(f"    MorphologyMap    : {type(g.morphology).__name__}")
    print(f"    AgreementChecker : {type(g.agreement).__name__}")

## 3. K-12 Curriculum Data — Built-in Word Lists

Every word list below is hardcoded — no external datasets needed.  
Each grade level includes:
- **Vocabulary words** (sight words → academic vocab)
- **Curriculum concepts** (domain-tagged: math, science, ELA, social studies)
- **Contrast pairs** (SAME and DIFFERENT relationships)

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#   K-12 PROGRESSIVE CURRICULUM — BUILT-IN WORD LISTS
# ══════════════════════════════════════════════════════════════════════
#
# Each grade is a dict with:
#   'words'    : list of vocabulary words to place on M(t)
#   'concepts' : list of (domain, concept_name) tuples
#   'same'     : list of (word1, word2) SAME-meaning pairs
#   'different' : list of (word1, word2) DIFFERENT-meaning pairs

K12_CURRICULUM = OrderedDict()

# ━━━ KINDERGARTEN ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
K12_CURRICULUM['K'] = {
    'label': 'Kindergarten',
    'words': [
        # Dolch Pre-Primer sight words
        'the', 'to', 'and', 'a', 'I', 'you', 'it', 'in', 'said', 'for',
        'up', 'look', 'is', 'go', 'we', 'little', 'down', 'can', 'see',
        'not', 'one', 'my', 'me', 'big', 'come', 'blue', 'red', 'where',
        'jump', 'away', 'here', 'help', 'make', 'find', 'run', 'play',
        'funny', 'three', 'two', 'yellow',
        # Colors
        'green', 'orange', 'purple', 'pink', 'black', 'white', 'brown',
        # Shapes
        'circle', 'square', 'triangle', 'rectangle', 'star',
        # Numbers (as words)
        'zero', 'four', 'five', 'six', 'seven', 'eight', 'nine', 'ten',
        # Body
        'hand', 'foot', 'head', 'eye', 'ear', 'nose', 'mouth',
        # Family / social
        'mom', 'dad', 'sister', 'brother', 'friend', 'teacher', 'school',
        # Actions
        'eat', 'sleep', 'walk', 'sit', 'stand', 'sing', 'draw', 'read',
        # Animals
        'cat', 'dog', 'bird', 'fish', 'bear', 'rabbit',
        # Nature
        'sun', 'moon', 'tree', 'flower', 'water', 'rain',
    ],
    'concepts': [
        ('math', 'counting'), ('math', 'shapes'), ('math', 'more'),
        ('math', 'less'), ('math', 'same'), ('math', 'different'),
        ('science', 'weather'), ('science', 'seasons'), ('science', 'animals'),
        ('science', 'plants'), ('science', 'senses'),
        ('ela', 'letters'), ('ela', 'sounds'), ('ela', 'rhyming'),
        ('ela', 'story'), ('ela', 'reading'),
        ('social', 'family'), ('social', 'community'), ('social', 'rules'),
        ('social', 'sharing'), ('social', 'kindness'),
    ],
    'same': [
        ('big', 'large'), ('little', 'small'), ('run', 'go'),
        ('happy', 'glad'), ('look', 'see'), ('friend', 'buddy'),
    ],
    'different': [
        ('big', 'little'), ('up', 'down'), ('run', 'sit'),
        ('hot', 'cold'), ('day', 'night'), ('happy', 'sad'),
        ('black', 'white'), ('come', 'go'),
    ],
}

# ━━━ GRADE 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
K12_CURRICULUM['1'] = {
    'label': 'Grade 1',
    'words': [
        # Dolch First Grade
        'after', 'again', 'an', 'any', 'ask', 'by', 'could', 'every',
        'fly', 'from', 'give', 'going', 'had', 'has', 'her', 'him',
        'his', 'how', 'just', 'know', 'let', 'live', 'may', 'of',
        'old', 'once', 'open', 'over', 'put', 'round', 'some', 'stop',
        'take', 'thank', 'them', 'then', 'think', 'walk', 'were', 'when',
        # Expanded academic
        'add', 'subtract', 'number', 'count', 'equal', 'group',
        'beginning', 'middle', 'end', 'sentence', 'word', 'letter',
        'question', 'answer', 'tell', 'write',
        # Science
        'push', 'pull', 'plant', 'seed', 'grow', 'leaf', 'root',
        'light', 'sound', 'heat', 'cold', 'hot', 'warm',
        # Social Studies
        'map', 'flag', 'country', 'city', 'people', 'work', 'home',
    ],
    'concepts': [
        ('math', 'addition'), ('math', 'subtraction'), ('math', 'place_value'),
        ('math', 'measurement'), ('math', 'comparison'),
        ('science', 'force'), ('science', 'push_pull'), ('science', 'life_cycle'),
        ('science', 'habitats'), ('science', 'matter'),
        ('ela', 'sentence'), ('ela', 'question'), ('ela', 'main_idea'),
        ('ela', 'characters'), ('ela', 'setting'),
        ('social', 'maps'), ('social', 'symbols'), ('social', 'needs_wants'),
        ('social', 'citizenship'), ('social', 'traditions'),
    ],
    'same': [
        ('add', 'plus'), ('home', 'house'), ('cold', 'cool'),
        ('begin', 'start'), ('tell', 'say'), ('work', 'job'),
    ],
    'different': [
        ('add', 'subtract'), ('push', 'pull'), ('hot', 'cold'),
        ('begin', 'end'), ('question', 'answer'), ('light', 'dark'),
        ('old', 'new'), ('open', 'close'),
    ],
}

# ━━━ GRADE 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
K12_CURRICULUM['2'] = {
    'label': 'Grade 2',
    'words': [
        # Dolch Second Grade
        'always', 'around', 'because', 'been', 'before', 'best', 'both',
        'buy', 'call', 'does', 'fast', 'first', 'found', 'gave',
        'goes', 'green', 'its', 'many', 'off', 'or', 'pull', 'read',
        'right', 'sing', 'sit', 'sleep', 'tell', 'their', 'these',
        'those', 'upon', 'us', 'use', 'very', 'wash', 'which', 'why',
        'wish', 'work', 'would', 'write',
        # Academic
        'describe', 'explain', 'compare', 'order', 'sort',
        'measure', 'graph', 'pattern', 'repeat', 'change',
        # Science
        'insect', 'mammal', 'reptile', 'amphibian', 'habitat',
        'weather', 'temperature', 'cloud', 'wind', 'storm',
        # Social Studies
        'state', 'president', 'vote', 'law', 'freedom',
        'history', 'past', 'present', 'future', 'calendar',
    ],
    'concepts': [
        ('math', 'skip_counting'), ('math', 'money'), ('math', 'time'),
        ('math', 'graphs'), ('math', 'patterns'),
        ('science', 'insects'), ('science', 'vertebrates'), ('science', 'weather_patterns'),
        ('science', 'solids_liquids_gases'), ('science', 'earth_materials'),
        ('ela', 'nouns'), ('ela', 'verbs'), ('ela', 'adjectives'),
        ('ela', 'sequence'), ('ela', 'retelling'),
        ('social', 'government'), ('social', 'timeline'), ('social', 'geography_basics'),
        ('social', 'culture'), ('social', 'economics_basics'),
    ],
    'same': [
        ('fast', 'quick'), ('many', 'several'), ('buy', 'purchase'),
        ('describe', 'explain'), ('pattern', 'repeat'), ('wish', 'hope'),
    ],
    'different': [
        ('fast', 'slow'), ('first', 'last'), ('always', 'never'),
        ('past', 'future'), ('buy', 'sell'), ('best', 'worst'),
        ('right', 'wrong'), ('freedom', 'captivity'),
    ],
}

# ━━━ GRADE 3 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
K12_CURRICULUM['3'] = {
    'label': 'Grade 3',
    'words': [
        # Academic vocabulary
        'multiply', 'divide', 'fraction', 'whole', 'product', 'quotient',
        'estimate', 'regroup', 'array', 'equation',
        'paragraph', 'topic', 'detail', 'opinion', 'fact',
        'continent', 'ocean', 'mountain', 'river', 'valley',
        # Science
        'ecosystem', 'organism', 'environment', 'adapt', 'survive',
        'energy', 'vibration', 'magnet', 'attract', 'repel',
        # ELA
        'character', 'setting', 'plot', 'problem', 'solution',
        'prefix', 'suffix', 'root', 'dictionary', 'definition',
        # Social Studies
        'geography', 'resource', 'natural', 'human', 'capital',
        'supply', 'demand', 'trade', 'import', 'export',
    ],
    'concepts': [
        ('math', 'multiplication'), ('math', 'division'), ('math', 'fractions'),
        ('math', 'area'), ('math', 'arrays'),
        ('science', 'ecosystems'), ('science', 'adaptation'), ('science', 'magnets'),
        ('science', 'sound_waves'), ('science', 'food_chains'),
        ('ela', 'plot_structure'), ('ela', 'point_of_view'), ('ela', 'text_features'),
        ('ela', 'prefixes_suffixes'), ('ela', 'cause_effect'),
        ('social', 'natural_resources'), ('social', 'supply_demand'),
        ('social', 'trade'), ('social', 'map_skills'), ('social', 'regions'),
    ],
    'same': [
        ('multiply', 'times'), ('whole', 'entire'), ('adapt', 'adjust'),
        ('environment', 'habitat'), ('trade', 'exchange'), ('opinion', 'belief'),
    ],
    'different': [
        ('multiply', 'divide'), ('fact', 'opinion'), ('attract', 'repel'),
        ('problem', 'solution'), ('import', 'export'), ('supply', 'demand'),
        ('natural', 'artificial'), ('survive', 'perish'),
    ],
}

# ━━━ GRADE 4 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
K12_CURRICULUM['4'] = {
    'label': 'Grade 4',
    'words': [
        'decimal', 'numerator', 'denominator', 'equivalent', 'improper',
        'perimeter', 'symmetry', 'angle', 'parallel', 'perpendicular',
        # Science
        'circuit', 'conductor', 'insulator', 'electricity', 'current',
        'erosion', 'weathering', 'mineral', 'rock', 'fossil',
        # ELA
        'theme', 'summary', 'inference', 'evidence', 'context',
        'narrative', 'dialogue', 'figurative', 'simile', 'metaphor',
        # Social Studies
        'colony', 'independence', 'revolution', 'constitution', 'democracy',
        'exploration', 'settlement', 'territory', 'treaty', 'amendment',
    ],
    'concepts': [
        ('math', 'decimals'), ('math', 'equivalent_fractions'), ('math', 'angles'),
        ('math', 'perimeter_area'), ('math', 'symmetry'),
        ('science', 'circuits'), ('science', 'electricity'),
        ('science', 'erosion_weathering'), ('science', 'rocks_minerals'),
        ('science', 'fossils'),
        ('ela', 'theme'), ('ela', 'inference'), ('ela', 'figurative_language'),
        ('ela', 'narrative_writing'), ('ela', 'evidence_based_writing'),
        ('social', 'american_revolution'), ('social', 'colonial_era'),
        ('social', 'constitution'), ('social', 'exploration'),
        ('social', 'indigenous_peoples'),
    ],
    'same': [
        ('evidence', 'proof'), ('democracy', 'republic'), ('theme', 'message'),
        ('equivalent', 'equal'), ('narrative', 'story'), ('summary', 'overview'),
    ],
    'different': [
        ('conductor', 'insulator'), ('parallel', 'perpendicular'),
        ('erosion', 'deposition'), ('numerator', 'denominator'),
        ('colony', 'independence'), ('simile', 'metaphor'),
        ('revolution', 'peace'), ('figurative', 'literal'),
    ],
}

# ━━━ GRADE 5 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
K12_CURRICULUM['5'] = {
    'label': 'Grade 5',
    'words': [
        'volume', 'coordinate', 'exponent', 'expression', 'variable',
        'classify', 'quadrilateral', 'polygon', 'prism', 'pyramid',
        # Science
        'cell', 'nucleus', 'membrane', 'tissue', 'organ',
        'photosynthesis', 'chlorophyll', 'oxygen', 'carbon', 'nitrogen',
        # ELA
        'conflict', 'resolution', 'narrator', 'perspective', 'mood',
        'essay', 'argument', 'claim', 'counterclaim', 'conclusion',
        # Social Studies
        'civilization', 'empire', 'culture', 'migration', 'agriculture',
        'monarchy', 'republic', 'citizen', 'economy', 'taxation',
    ],
    'concepts': [
        ('math', 'volume'), ('math', 'coordinates'), ('math', 'expressions'),
        ('math', 'order_of_operations'), ('math', 'geometry_classification'),
        ('science', 'cell_biology'), ('science', 'photosynthesis'),
        ('science', 'matter_properties'), ('science', 'earths_systems'),
        ('science', 'ecosystems_energy'),
        ('ela', 'story_elements'), ('ela', 'argumentative_writing'),
        ('ela', 'research'), ('ela', 'poetry'), ('ela', 'drama'),
        ('social', 'ancient_civilizations'), ('social', 'westward_expansion'),
        ('social', 'early_republic'), ('social', 'civil_war_era'),
        ('social', 'economics'),
    ],
    'same': [
        ('conflict', 'struggle'), ('conclusion', 'ending'), ('citizen', 'resident'),
        ('classify', 'categorize'), ('argument', 'claim'), ('migration', 'movement'),
    ],
    'different': [
        ('conflict', 'resolution'), ('claim', 'counterclaim'),
        ('monarchy', 'republic'), ('oxygen', 'carbon'),
        ('agriculture', 'industry'), ('perspective', 'fact'),
        ('cell', 'organ'), ('empire', 'colony'),
    ],
}

# ━━━ GRADE 6 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
K12_CURRICULUM['6'] = {
    'label': 'Grade 6',
    'words': [
        'ratio', 'proportion', 'percent', 'integer', 'absolute',
        'inequality', 'dependent', 'independent', 'statistics', 'median',
        # Science
        'hypothesis', 'experiment', 'variable', 'observation', 'conclusion',
        'element', 'compound', 'mixture', 'density', 'mass',
        # ELA
        'allegory', 'irony', 'symbolism', 'tone', 'author',
        'rhetoric', 'persuasion', 'thesis', 'citation', 'source',
        # Social Studies
        'civilization', 'dynasty', 'theocracy', 'polytheism', 'monotheism',
        'agriculture', 'irrigation', 'surplus', 'specialization', 'barter',
    ],
    'concepts': [
        ('math', 'ratios_proportions'), ('math', 'integers'),
        ('math', 'statistics_probability'), ('math', 'inequalities'),
        ('math', 'geometry_nets'),
        ('science', 'scientific_method'), ('science', 'elements_compounds'),
        ('science', 'density'), ('science', 'cells_organisms'),
        ('science', 'earth_history'),
        ('ela', 'literary_analysis'), ('ela', 'persuasive_writing'),
        ('ela', 'research_citations'), ('ela', 'irony_symbolism'),
        ('ela', 'author_purpose'),
        ('social', 'ancient_river_civilizations'), ('social', 'classical_greece'),
        ('social', 'roman_empire'), ('social', 'early_religions'),
        ('social', 'trade_routes'),
    ],
    'same': [
        ('hypothesis', 'prediction'), ('observation', 'evidence'),
        ('ratio', 'proportion'), ('rhetoric', 'persuasion'),
        ('thesis', 'claim'), ('surplus', 'excess'),
    ],
    'different': [
        ('hypothesis', 'conclusion'), ('dependent', 'independent'),
        ('element', 'compound'), ('polytheism', 'monotheism'),
        ('integer', 'fraction'), ('allegory', 'literal'),
        ('barter', 'currency'), ('density', 'volume'),
    ],
}

# ━━━ GRADE 7 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
K12_CURRICULUM['7'] = {
    'label': 'Grade 7',
    'words': [
        'coefficient', 'constant', 'linear', 'slope', 'intercept',
        'probability', 'sample', 'population', 'mean', 'deviation',
        # Science
        'mitosis', 'chromosome', 'gene', 'trait', 'heredity',
        'atom', 'molecule', 'proton', 'neutron', 'electron',
        # ELA
        'connotation', 'denotation', 'syntax', 'diction', 'voice',
        'analogy', 'allusion', 'foreshadowing', 'flashback', 'motif',
        # Social Studies
        'feudalism', 'reformation', 'renaissance', 'enlightenment', 'mercantilism',
        'absolutism', 'parliament', 'charter', 'crusade', 'plague',
    ],
    'concepts': [
        ('math', 'linear_equations'), ('math', 'slope_intercept'),
        ('math', 'probability'), ('math', 'statistical_inference'),
        ('math', 'proportional_relationships'),
        ('science', 'cell_division'), ('science', 'genetics'),
        ('science', 'atomic_structure'), ('science', 'chemical_bonds'),
        ('science', 'forces_motion'),
        ('ela', 'literary_devices'), ('ela', 'analytical_writing'),
        ('ela', 'connotation_denotation'), ('ela', 'narrative_techniques'),
        ('ela', 'speech_debate'),
        ('social', 'medieval_period'), ('social', 'renaissance_reformation'),
        ('social', 'age_of_exploration'), ('social', 'early_modern_period'),
        ('social', 'comparative_government'),
    ],
    'same': [
        ('gene', 'heredity'), ('atom', 'particle'), ('slope', 'rate'),
        ('renaissance', 'rebirth'), ('connotation', 'implication'),
        ('reformation', 'transformation'),
    ],
    'different': [
        ('connotation', 'denotation'), ('proton', 'electron'),
        ('feudalism', 'democracy'), ('foreshadowing', 'flashback'),
        ('linear', 'nonlinear'), ('mitosis', 'meiosis'),
        ('constant', 'variable'), ('absolutism', 'parliament'),
    ],
}

# ━━━ GRADE 8 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
K12_CURRICULUM['8'] = {
    'label': 'Grade 8',
    'words': [
        'function', 'transformation', 'rotation', 'reflection', 'translation',
        'irrational', 'scientific_notation', 'bivariate', 'correlation', 'regression',
        # Science
        'velocity', 'acceleration', 'momentum', 'friction', 'gravity',
        'wavelength', 'frequency', 'amplitude', 'spectrum', 'radiation',
        # ELA
        'satire', 'parody', 'paradox', 'juxtaposition', 'antithesis',
        'propaganda', 'bias', 'credibility', 'ethos', 'pathos',
        # Social Studies
        'constitution', 'federalism', 'amendment', 'ratification', 'sovereignty',
        'industrialization', 'abolition', 'suffrage', 'manifest_destiny', 'reconstruction',
    ],
    'concepts': [
        ('math', 'functions'), ('math', 'transformations'),
        ('math', 'pythagorean_theorem'), ('math', 'scientific_notation'),
        ('math', 'linear_models'),
        ('science', 'newtons_laws'), ('science', 'waves_energy'),
        ('science', 'electromagnetic_spectrum'), ('science', 'motion_forces'),
        ('science', 'conservation_energy'),
        ('ela', 'rhetoric_analysis'), ('ela', 'satire_parody'),
        ('ela', 'research_argument'), ('ela', 'media_literacy'),
        ('ela', 'comparative_literature'),
        ('social', 'us_constitution'), ('social', 'civil_war'),
        ('social', 'industrial_revolution'), ('social', 'westward_movement'),
        ('social', 'reform_movements'),
    ],
    'same': [
        ('velocity', 'speed'), ('bias', 'prejudice'), ('amendment', 'change'),
        ('sovereignty', 'independence'), ('suffrage', 'voting'),
        ('propaganda', 'persuasion'),
    ],
    'different': [
        ('velocity', 'acceleration'), ('rotation', 'reflection'),
        ('satire', 'praise'), ('abolition', 'slavery'),
        ('ethos', 'pathos'), ('correlation', 'causation'),
        ('friction', 'momentum'), ('federalism', 'sovereignty'),
    ],
}

# ━━━ GRADE 9 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
K12_CURRICULUM['9'] = {
    'label': 'Grade 9',
    'words': [
        'polynomial', 'quadratic', 'factoring', 'domain', 'range',
        'exponential', 'radical', 'rational', 'sequence', 'arithmetic',
        # Science (Biology)
        'evolution', 'natural_selection', 'adaptation', 'speciation', 'mutation',
        'ecosystem', 'biodiversity', 'symbiosis', 'homeostasis', 'metabolism',
        # ELA
        'protagonist', 'antagonist', 'archetype', 'dystopia', 'soliloquy',
        'tragedy', 'comedy', 'epic', 'sonnet', 'meter',
        # Social Studies (World History)
        'imperialism', 'nationalism', 'colonialism', 'revolution', 'ideology',
        'totalitarianism', 'fascism', 'communism', 'capitalism', 'liberalism',
    ],
    'concepts': [
        ('math', 'algebra_1'), ('math', 'polynomials'), ('math', 'quadratics'),
        ('math', 'exponential_functions'), ('math', 'sequences_series'),
        ('science', 'evolution_natural_selection'), ('science', 'ecology'),
        ('science', 'cell_processes'), ('science', 'genetics_heredity'),
        ('science', 'biodiversity'),
        ('ela', 'world_literature'), ('ela', 'dramatic_forms'),
        ('ela', 'poetry_analysis'), ('ela', 'argumentative_essay'),
        ('ela', 'greek_tragedy'),
        ('social', 'world_war_1'), ('social', 'russian_revolution'),
        ('social', 'imperialism_colonialism'), ('social', 'ideologies'),
        ('social', 'industrial_society'),
    ],
    'same': [
        ('evolution', 'adaptation'), ('protagonist', 'hero'),
        ('nationalism', 'patriotism'), ('capitalism', 'free_market'),
        ('metabolism', 'process'), ('ideology', 'belief_system'),
    ],
    'different': [
        ('protagonist', 'antagonist'), ('tragedy', 'comedy'),
        ('capitalism', 'communism'), ('imperialism', 'independence'),
        ('domain', 'range'), ('symbiosis', 'competition'),
        ('fascism', 'liberalism'), ('mutation', 'stability'),
    ],
}

# ━━━ GRADE 10 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
K12_CURRICULUM['10'] = {
    'label': 'Grade 10',
    'words': [
        'logarithm', 'asymptote', 'matrix', 'vector', 'theorem',
        'trigonometry', 'sine', 'cosine', 'tangent', 'radian',
        # Science (Chemistry)
        'stoichiometry', 'mole', 'covalent', 'ionic', 'catalyst',
        'equilibrium', 'reaction', 'oxidation', 'reduction', 'pH',
        # ELA
        'existentialism', 'modernism', 'stream_of_consciousness', 'unreliable_narrator', 'bildungsroman',
        'rhetoric', 'ethos', 'logos', 'pathos', 'kairos',
        # Social Studies (US History / Government)
        'hegemony', 'containment', 'detente', 'proxy_war', 'cold_war',
        'civil_rights', 'segregation', 'integration', 'activism', 'legislation',
    ],
    'concepts': [
        ('math', 'geometry_proofs'), ('math', 'trigonometry'),
        ('math', 'logarithms'), ('math', 'matrices_vectors'),
        ('math', 'circle_theorems'),
        ('science', 'chemical_reactions'), ('science', 'stoichiometry'),
        ('science', 'bonding'), ('science', 'acids_bases'),
        ('science', 'thermochemistry'),
        ('ela', 'modernist_literature'), ('ela', 'rhetorical_analysis'),
        ('ela', 'seminar_discussion'), ('ela', 'existentialist_texts'),
        ('ela', 'comparative_essay'),
        ('social', 'cold_war'), ('social', 'civil_rights_movement'),
        ('social', 'vietnam_era'), ('social', 'post_ww2_america'),
        ('social', 'globalization_beginnings'),
    ],
    'same': [
        ('catalyst', 'accelerator'), ('equilibrium', 'balance'),
        ('segregation', 'separation'), ('activism', 'advocacy'),
        ('rhetoric', 'persuasion'), ('containment', 'restriction'),
    ],
    'different': [
        ('covalent', 'ionic'), ('oxidation', 'reduction'),
        ('segregation', 'integration'), ('sine', 'cosine'),
        ('logarithm', 'exponent'), ('modernism', 'traditionalism'),
        ('detente', 'cold_war'), ('ethos', 'logos'),
    ],
}

# ━━━ GRADE 11 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
K12_CURRICULUM['11'] = {
    'label': 'Grade 11',
    'words': [
        'derivative', 'integral', 'limit', 'continuity', 'convergence',
        'differentiation', 'antiderivative', 'series', 'infinite', 'tangent_line',
        # Science (Physics)
        'torque', 'angular_momentum', 'centripetal', 'gravitational_field', 'potential_energy',
        'kinetic_energy', 'conservation', 'impulse', 'work', 'power',
        # ELA (American Literature)
        'transcendentalism', 'romanticism', 'realism', 'naturalism', 'harlem_renaissance',
        'american_dream', 'manifest_destiny', 'individualism', 'conformity', 'dissent',
        # Social Studies (US History)
        'progressivism', 'isolationism', 'interventionism', 'new_deal', 'great_depression',
        'prohibition', 'urbanization', 'immigration', 'assimilation', 'nativism',
    ],
    'concepts': [
        ('math', 'calculus_limits'), ('math', 'derivatives'),
        ('math', 'integrals'), ('math', 'series_convergence'),
        ('math', 'applications_of_calculus'),
        ('science', 'classical_mechanics'), ('science', 'energy_conservation'),
        ('science', 'rotational_motion'), ('science', 'gravitational_fields'),
        ('science', 'work_energy_theorem'),
        ('ela', 'american_literary_movements'), ('ela', 'transcendentalist_texts'),
        ('ela', 'harlem_renaissance_poetry'), ('ela', 'american_novel'),
        ('ela', 'synthesis_essay'),
        ('social', 'progressive_era'), ('social', 'world_war_2'),
        ('social', 'great_depression_new_deal'), ('social', 'immigration_patterns'),
        ('social', 'modern_america'),
    ],
    'same': [
        ('derivative', 'rate_of_change'), ('integral', 'area_under_curve'),
        ('conservation', 'preservation'), ('dissent', 'protest'),
        ('immigration', 'migration'), ('progressivism', 'reform'),
    ],
    'different': [
        ('derivative', 'integral'), ('kinetic_energy', 'potential_energy'),
        ('romanticism', 'realism'), ('isolationism', 'interventionism'),
        ('individualism', 'conformity'), ('convergence', 'divergence'),
        ('assimilation', 'nativism'), ('limit', 'infinite'),
    ],
}

# ━━━ GRADE 12 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
K12_CURRICULUM['12'] = {
    'label': 'Grade 12',
    'words': [
        'multivariable', 'differential_equation', 'eigenvalue', 'proof',
        'axiom', 'conjecture', 'topology', 'manifold', 'isomorphism', 'cardinality',
        # Science (AP Physics / Chemistry)
        'entropy', 'enthalpy', 'thermodynamics', 'quantum', 'relativity',
        'electromagnetic', 'wavefunction', 'uncertainty_principle', 'superposition', 'interference',
        # ELA (World Literature / AP)
        'postmodernism', 'deconstruction', 'intertextuality', 'metafiction', 'magical_realism',
        'dialectic', 'epistemology', 'ontology', 'phenomenology', 'hermeneutics',
        # Social Studies (Government / Economics / Global)
        'geopolitics', 'diplomacy', 'multilateralism', 'sovereignty', 'globalization',
        'fiscal_policy', 'monetary_policy', 'inflation', 'macroeconomics', 'microeconomics',
    ],
    'concepts': [
        ('math', 'multivariable_calculus'), ('math', 'differential_equations'),
        ('math', 'linear_algebra'), ('math', 'mathematical_proofs'),
        ('math', 'abstract_mathematics'),
        ('science', 'thermodynamics'), ('science', 'quantum_mechanics'),
        ('science', 'relativity'), ('science', 'electromagnetic_theory'),
        ('science', 'modern_physics'),
        ('ela', 'postmodern_literature'), ('ela', 'literary_theory'),
        ('ela', 'philosophical_texts'), ('ela', 'critical_theory'),
        ('ela', 'capstone_research'),
        ('social', 'international_relations'), ('social', 'macroeconomic_policy'),
        ('social', 'global_governance'), ('social', 'political_philosophy'),
        ('social', 'contemporary_issues'),
    ],
    'same': [
        ('entropy', 'disorder'), ('quantum', 'discrete'),
        ('axiom', 'postulate'), ('diplomacy', 'negotiation'),
        ('globalization', 'interconnection'), ('epistemology', 'knowledge_theory'),
    ],
    'different': [
        ('entropy', 'order'), ('quantum', 'classical'),
        ('macroeconomics', 'microeconomics'), ('inflation', 'deflation'),
        ('postmodernism', 'modernism'), ('fiscal_policy', 'monetary_policy'),
        ('superposition', 'collapse'), ('multilateralism', 'unilateralism'),
    ],
}

# ── Summary ────────────────────────────────────────────────────────────
total_words = 0
total_concepts = 0
total_pairs = 0
print("\u2550" * 70)
print("  K-12 CURRICULUM SUMMARY")
print("\u2550" * 70)
for grade, data in K12_CURRICULUM.items():
    nw = len(data['words'])
    nc = len(data['concepts'])
    ns = len(data['same'])
    nd = len(data['different'])
    total_words += nw
    total_concepts += nc
    total_pairs += ns + nd
    print(f"  {data['label']:15s}  words={nw:3d}  concepts={nc:2d}  "
          f"same={ns:2d}  diff={nd:2d}")
print(f"{'\u2500' * 70}")
print(f"  TOTAL            words={total_words:3d}  concepts={total_concepts:3d}  "
      f"pairs={total_pairs:3d}")
print(f"\u2550" * 70)

## 4. Progressive Pipeline — K through 12

Each grade level:
1. **Places vocabulary words** on M(t) via C3 Annealing
2. **Places curriculum concepts** with domain tags
3. **Runs contrast pairs** (SAME pulls together, DIFFERENT pushes apart)
4. **Reports progress** — concepts added, temperature, time

The manifold grows progressively — each grade builds on all prior knowledge.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#   PROGRESSIVE K-12 FEEDING PIPELINE
# ══════════════════════════════════════════════════════════════════════

from src.vocabulary.word_placer import structural_feature_vector

grade_stats = OrderedDict()  # track per-grade metrics
t_pipeline_start = time.time()

for grade_key, data in K12_CURRICULUM.items():
    t_grade = time.time()
    grade_label = data['label']
    n_before = pipeline.n_concepts
    
    print(f"\n{'\u2550' * 70}")
    print(f"  GRADE {grade_key} — {grade_label}")
    print(f"{'\u2550' * 70}")
    print(f"  Manifold before : {n_before:,} concepts  |  T = {pipeline.temperature:.6f}")
    
    # ── 4A: Place vocabulary words ────────────────────────────────────
    n_placed = 0
    n_skipped = 0
    for word in data['words']:
        label = f"k12::{grade_key}::{word}"
        # Build structural vector for this word
        vec = structural_feature_vector(word)
        try:
            exp = Experience(vector=vec, label=label)
            result = pipeline.learn(exp)
            n_placed += 1
        except Exception as e:
            n_skipped += 1
    
    print(f"  Words placed    : {n_placed:3d}  (skipped: {n_skipped})")
    
    # ── 4B: Place curriculum concepts ─────────────────────────────────
    n_concepts_placed = 0
    for domain, concept_name in data['concepts']:
        label = f"k12::{grade_key}::{domain}::{concept_name}"
        # Generate vector from concept name
        vec = structural_feature_vector(concept_name.replace('_', ' '))
        try:
            exp = Experience(vector=vec, label=label)
            result = pipeline.learn(exp)
            n_concepts_placed += 1
        except Exception:
            pass
    
    print(f"  Concepts placed : {n_concepts_placed:3d}")
    
    # ── 4C: Contrast pairs ────────────────────────────────────────────
    n_same = 0
    n_diff = 0
    
    # SAME pairs — pull semantically similar words together
    for w1, w2 in data['same']:
        l1 = f"k12::{grade_key}::{w1}"
        l2 = f"k12::{grade_key}::{w2}"
        # Ensure both exist on manifold (place w2 if needed)
        if l2 not in pipeline.manifold._points:
            vec2 = structural_feature_vector(w2)
            pipeline.learn(Experience(vector=vec2, label=l2))
        if l1 in pipeline.manifold._points and l2 in pipeline.manifold._points:
            try:
                pipeline.contrast(l1, l2, 'same')
                n_same += 1
            except Exception:
                pass
    
    # DIFFERENT pairs — push semantically opposed words apart
    for w1, w2 in data['different']:
        l1 = f"k12::{grade_key}::{w1}"
        l2 = f"k12::{grade_key}::{w2}"
        if l2 not in pipeline.manifold._points:
            vec2 = structural_feature_vector(w2)
            pipeline.learn(Experience(vector=vec2, label=l2))
        if l1 in pipeline.manifold._points and l2 in pipeline.manifold._points:
            try:
                pipeline.contrast(l1, l2, 'different')
                n_diff += 1
            except Exception:
                pass
    
    print(f"  Contrast SAME   : {n_same:3d}  |  DIFFERENT: {n_diff:3d}")
    
    n_after = pipeline.n_concepts
    elapsed_grade = time.time() - t_grade
    
    grade_stats[grade_key] = {
        'label': grade_label,
        'words_placed': n_placed,
        'concepts_placed': n_concepts_placed,
        'same_pairs': n_same,
        'diff_pairs': n_diff,
        'concepts_before': n_before,
        'concepts_after': n_after,
        'temperature': pipeline.temperature,
        'elapsed': elapsed_grade,
    }
    
    print(f"  Manifold after  : {n_after:,} concepts  (+{n_after - n_before})")
    print(f"  Temperature     : {pipeline.temperature:.6f}")
    print(f"  Time            : {elapsed_grade:.2f}s")

total_pipeline_time = time.time() - t_pipeline_start

# ── Pipeline Summary ──────────────────────────────────────────────────
print(f"\n\n{'\u2550' * 70}")
print(f"  K-12 PROGRESSIVE PIPELINE COMPLETE")
print(f"{'\u2550' * 70}")
print(f"{'Grade':<12s} {'Words':>6s} {'Concepts':>9s} {'Same':>5s} {'Diff':>5s} {'Total':>7s} {'Time':>7s}")
print(f"{'\u2500' * 70}")
for gk, gs in grade_stats.items():
    print(f"{gs['label']:<12s} {gs['words_placed']:>6d} {gs['concepts_placed']:>9d} "
          f"{gs['same_pairs']:>5d} {gs['diff_pairs']:>5d} "
          f"{gs['concepts_after']:>7,} {gs['elapsed']:>6.1f}s")
print(f"{'\u2500' * 70}")
print(f"  Total concepts on M(t) : {pipeline.n_concepts:,}")
print(f"  Final temperature      : {pipeline.temperature:.6f}")
print(f"  Total pipeline time    : {total_pipeline_time:.1f}s ({total_pipeline_time/60:.1f} min)")
print(f"{'\u2550' * 70}")

## 5. Build Expression Entries for K-12 Vocabulary

Convert all placed K-12 words into `ExpressionEntry` objects  
so the C7 renderer can produce grade-appropriate language.

In [ ]:
# ── Build ExpressionEntry objects for all K-12 words ──────────────────
from src.phase1.expression.matcher import ExpressionEntry
from src.vocabulary.word_placer import structural_feature_vector

k12_entries = []
seen_texts = set()

# Grade-level register mapping
GRADE_REGISTER = {
    'K': 'simple', '1': 'simple', '2': 'simple',
    '3': 'neutral', '4': 'neutral', '5': 'neutral',
    '6': 'neutral', '7': 'neutral', '8': 'formal',
    '9': 'formal', '10': 'formal', '11': 'formal', '12': 'formal',
}

def _make_wave_profile(vec):
    """Normalize a 104D vector into a unit wave profile."""
    norm = np.linalg.norm(vec)
    if norm < 1e-12:
        return np.ones(104, dtype=float) / np.sqrt(104)
    return vec / norm

for grade_key, data in K12_CURRICULUM.items():
    register = GRADE_REGISTER[grade_key]
    all_grade_words = list(data['words'])
    
    # Also include words introduced via contrast pairs
    for w1, w2 in data['same'] + data['different']:
        if w2 not in all_grade_words:
            all_grade_words.append(w2)
    
    for word in all_grade_words:
        clean_word = word.replace('_', ' ')
        if clean_word in seen_texts:
            continue
        seen_texts.add(clean_word)
        
        label = f"k12::{grade_key}::{word}"
        
        # Get position vector for this word
        try:
            vec = pipeline.manifold.position(label)
        except (KeyError, ValueError):
            vec = structural_feature_vector(word)
        
        wave_profile = _make_wave_profile(vec)
        
        # Determine hedging from probabilistic fiber
        prob_fiber = vec[88:104]
        hedging = bool(float(np.mean(prob_fiber)) < 0.3)
        
        # Determine causal strength from causal fiber
        causal_fiber = vec[64:80]
        causal_strength = float(np.mean(np.abs(causal_fiber)))
        
        entry = ExpressionEntry(
            text=clean_word,
            wave_profile=wave_profile,
            register=register,
            rhythm='short' if len(clean_word) <= 5 else 'medium',
            hedging=hedging,
            causal_strength=causal_strength,
        )
        k12_entries.append(entry)
    
    # Concept entries (as phrases)
    for domain, concept_name in data['concepts']:
        clean_name = concept_name.replace('_', ' ')
        phrase = f"{domain}: {clean_name}"
        if phrase in seen_texts:
            continue
        seen_texts.add(phrase)
        
        label = f"k12::{grade_key}::{domain}::{concept_name}"
        try:
            vec = pipeline.manifold.position(label)
        except (KeyError, ValueError):
            vec = structural_feature_vector(concept_name.replace('_', ' '))
        
        wave_profile = _make_wave_profile(vec)
        causal_fiber = vec[64:80]
        causal_strength = float(np.mean(np.abs(causal_fiber)))
        
        entry = ExpressionEntry(
            text=phrase,
            wave_profile=wave_profile,
            register=register,
            rhythm='medium' if len(phrase) <= 20 else 'long',
            hedging=False,
            causal_strength=causal_strength,
        )
        k12_entries.append(entry)

print(f"\u2713 Built {len(k12_entries):,} K-12 expression entries")
print(f"  Register distribution:")
reg_dist = Counter(e.register for e in k12_entries)
for reg, cnt in reg_dist.most_common():
    print(f"    {reg:10s} : {cnt:>4,}")

# Load entries directly into the renderer's vocabulary list
# (load_vocabulary() expects a file path — we have in-memory entries)
pipeline._renderer.matcher.vocabulary.extend(k12_entries)
pipeline._renderer.matcher._faiss_dirty = True
print(f"\n\u2713 Loaded {len(k12_entries):,} entries into C7 renderer")
print(f"  Total vocabulary : {len(pipeline._renderer.matcher.vocabulary):,} entries")

## 6. Grade-Level Query Interface

Query the manifold at each grade band and observe how output complexity
scales with the progressive curriculum.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#   GRADE-LEVEL QUERIES — Progressive Complexity
# ══════════════════════════════════════════════════════════════════════
#
# For each grade band, we query a concept that was placed at that level
# and observe how the output reflects grade-appropriate language.

GRADE_QUERIES = [
    # (grade_key, label, description)
    ('K',  'k12::K::cat',           'What is a cat?'),
    ('K',  'k12::K::sun',           'What is the sun?'),
    ('2',  'k12::2::weather',       'What is weather?'),
    ('3',  'k12::3::ecosystem',     'What is an ecosystem?'),
    ('4',  'k12::4::electricity',   'What is electricity?'),
    ('5',  'k12::5::photosynthesis', 'What is photosynthesis?'),
    ('6',  'k12::6::hypothesis',    'What is a hypothesis?'),
    ('7',  'k12::7::mitosis',       'What is mitosis?'),
    ('8',  'k12::8::velocity',      'What is velocity?'),
    ('9',  'k12::9::evolution',     'What is evolution?'),
    ('10', 'k12::10::equilibrium',  'What is chemical equilibrium?'),
    ('11', 'k12::11::derivative',   'What is a derivative?'),
    ('12', 'k12::12::entropy',      'What is entropy?'),
]

query_results = []

print("\u2550" * 70)
print("  GRADE-LEVEL QUERY RESULTS")
print("\u2550" * 70)

for grade_key, label, description in GRADE_QUERIES:
    grade_name = K12_CURRICULUM[grade_key]['label']
    
    try:
        vec = pipeline.manifold.position(label)
    except (KeyError, ValueError):
        print(f"\n  [{grade_name}] {description}")
        print(f"    \u2718 Label '{label}' not on manifold")
        continue
    
    result = pipeline.query(vec, label=description)
    query_results.append((grade_key, grade_name, description, result))
    
    print(f"\n  [{grade_name}] {description}")
    print(f"    Steps: {result.n_steps}  |  Reason: {result.termination_reason}")
    print(f"    Confidence: {result.confidence:.3f}  |  Wave: {result.wave_confidence:.3f}")
    
    # Check if grammar was used
    grammar_used = any(
        d.get('grammar_enhanced', False) 
        for d in result.wave.points if hasattr(result, '_diagnostics')
    ) if hasattr(result, '_diagnostics') else 'unknown'
    
    print(f"    Output: {result.text[:150]}")
    print(f"  {'\u2500' * 60}")

print(f"\n\u2550" * 70)
print(f"  {len(query_results)} queries completed")
print(f"\u2550" * 70)

## 7. Cross-Grade Concept Queries

Ask the same concept at different grade levels to see
how language complexity scales with education level.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#   CROSS-GRADE COMPARISON — Same concept, different grade levels
# ══════════════════════════════════════════════════════════════════════
#
# Topics that appear across multiple grades (e.g. 'energy' from
# Grade 3 science through Grade 12 physics). We query each and
# compare the output complexity.

CROSS_GRADE_TOPICS = [
    {
        'name': 'Energy',
        'queries': [
            ('3', 'k12::3::energy',              'What is energy? (Grade 3)'),
            ('8', 'k12::8::science::conservation_energy', 'What is conservation of energy? (Grade 8)'),
            ('11', 'k12::11::kinetic_energy',    'What is kinetic energy? (Grade 11)'),
        ]
    },
    {
        'name': 'Government',
        'queries': [
            ('2', 'k12::2::social::government',  'What is government? (Grade 2)'),
            ('4', 'k12::4::democracy',           'What is democracy? (Grade 4)'),
            ('8', 'k12::8::federalism',          'What is federalism? (Grade 8)'),
            ('12', 'k12::12::sovereignty',       'What is sovereignty? (Grade 12)'),
        ]
    },
    {
        'name': 'Mathematics',
        'queries': [
            ('1', 'k12::1::add',                 'What is addition? (Grade 1)'),
            ('3', 'k12::3::multiply',            'What is multiplication? (Grade 3)'),
            ('6', 'k12::6::ratio',               'What is a ratio? (Grade 6)'),
            ('9', 'k12::9::polynomial',          'What is a polynomial? (Grade 9)'),
            ('11', 'k12::11::derivative',        'What is a derivative? (Grade 11)'),
        ]
    },
]

print("\u2550" * 70)
print("  CROSS-GRADE CONCEPT COMPARISON")
print("\u2550" * 70)

for topic in CROSS_GRADE_TOPICS:
    print(f"\n\u250c{'\u2500' * 68}\u2510")
    print(f"\u2502  Topic: {topic['name']:<58s}\u2502")
    print(f"\u2514{'\u2500' * 68}\u2518")
    
    for grade_key, label, description in topic['queries']:
        grade_name = K12_CURRICULUM[grade_key]['label']
        try:
            vec = pipeline.manifold.position(label)
            result = pipeline.query(vec, label=description)
            print(f"\n    [{grade_name:>13s}]  {description}")
            print(f"    {'':>13s}   \u2192 {result.text[:130]}")
            print(f"    {'':>13s}   conf={result.confidence:.3f}  steps={result.n_steps}")
        except Exception as e:
            print(f"\n    [{grade_name:>13s}]  \u2718 {label}: {e}")

## 8. Content Generation — Grade-Appropriate Explanations

Generate multi-sentence explanations by querying sequences of related
concepts within a grade band.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#   CONTENT GENERATION — Multi-concept explanations
# ══════════════════════════════════════════════════════════════════════

def generate_explanation(pipeline, concept_labels, title="Explanation"):
    """Query multiple related concepts and compose their outputs."""
    sentences = []
    for label in concept_labels:
        try:
            vec = pipeline.manifold.position(label)
            result = pipeline.query(vec, label=label)
            if result.text and len(result.text.strip()) > 5:
                sentences.append(result.text.strip())
        except (KeyError, ValueError):
            continue
    return ' '.join(sentences)


EXPLANATIONS = [
    {
        'title': 'Grade 3 — How Ecosystems Work',
        'labels': [
            'k12::3::ecosystem', 'k12::3::organism',
            'k12::3::environment', 'k12::3::adapt', 'k12::3::survive',
        ],
    },
    {
        'title': 'Grade 5 — Cells and Photosynthesis',
        'labels': [
            'k12::5::cell', 'k12::5::nucleus', 'k12::5::membrane',
            'k12::5::photosynthesis', 'k12::5::chlorophyll', 'k12::5::oxygen',
        ],
    },
    {
        'title': 'Grade 8 — Forces and Motion',
        'labels': [
            'k12::8::velocity', 'k12::8::acceleration', 'k12::8::momentum',
            'k12::8::friction', 'k12::8::gravity',
        ],
    },
    {
        'title': 'Grade 10 — Chemical Reactions',
        'labels': [
            'k12::10::catalyst', 'k12::10::equilibrium', 'k12::10::reaction',
            'k12::10::oxidation', 'k12::10::reduction',
        ],
    },
    {
        'title': 'Grade 12 — Quantum and Thermodynamics',
        'labels': [
            'k12::12::entropy', 'k12::12::thermodynamics',
            'k12::12::quantum', 'k12::12::superposition',
            'k12::12::uncertainty_principle',
        ],
    },
]

print("\u2550" * 70)
print("  CONTENT GENERATION — Grade-Appropriate Explanations")
print("\u2550" * 70)

for exp in EXPLANATIONS:
    text = generate_explanation(pipeline, exp['labels'], exp['title'])
    print(f"\n  \u250c {'\u2500' * 66} \u2510")
    print(f"  \u2502 {exp['title']:<66s} \u2502")
    print(f"  \u2514 {'\u2500' * 66} \u2518")
    # Word-wrap at ~70 chars
    import textwrap
    wrapped = textwrap.fill(text, width=66, initial_indent='    ', subsequent_indent='    ')
    print(wrapped)
    print(f"    [{len(text)} chars, {len(text.split())} words]")

print(f"\n{'\u2550' * 70}")

## 9. Evaluation Suite

In [ ]:
# ── Run evaluation on K-12 concepts ──────────────────────────────────
from src.phase5 import PipelineEvaluator

evaluator = PipelineEvaluator(pipeline)

# Gather eval vectors from across all grade levels
eval_labels = []
eval_vecs = []

for grade_key, data in K12_CURRICULUM.items():
    # Sample 3 words per grade
    for word in data['words'][:3]:
        label = f"k12::{grade_key}::{word}"
        try:
            vec = pipeline.manifold.position(label)
            eval_labels.append(label)
            eval_vecs.append(vec)
        except (KeyError, ValueError):
            continue

print(f"Running evaluation suite with {len(eval_vecs)} queries...")
t0 = time.time()
suite = evaluator.run_suite(eval_vecs, eval_labels)
elapsed = time.time() - t0

print(f"\n\u2713 Evaluation complete ({elapsed:.1f}s)")
print(f"  Mean coherence       : {suite.mean_coherence:.4f}")
print(f"  Novelty is decaying  : {suite.novelty_is_decaying}")
print(f"  Termination reasons  :")
for reason, count in suite.termination_distribution.items():
    print(f"    {reason:30s} : {count}")

## 10. Save Artifacts

In [ ]:
# ── Save K-12 model artifacts ────────────────────────────────────────
OUTPUT_DIR = "/kaggle/working"
K12_MANIFOLD_PATH = f"{OUTPUT_DIR}/FLOW-K12_manifold.npz"
K12_VOCAB_PATH    = f"{OUTPUT_DIR}/FLOW-K12_vocab.npz"

# Ensure output dir exists (always true on Kaggle, but just in case)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save manifold
n_saved = ManifoldSnapshot.save(pipeline.manifold, K12_MANIFOLD_PATH)
print(f"\u2713 Manifold saved: {K12_MANIFOLD_PATH}")
print(f"  Concepts : {n_saved:,}")
print(f"  Size     : {os.path.getsize(K12_MANIFOLD_PATH) / 1024:.1f} KB")

# Save vocabulary
n_entries = VocabularyStore.save(k12_entries, K12_VOCAB_PATH)
print(f"\n\u2713 Vocabulary saved: {K12_VOCAB_PATH}")
print(f"  Entries  : {n_entries:,}")
print(f"  Size     : {os.path.getsize(K12_VOCAB_PATH) / 1024:.1f} KB")

# ── Push to GitHub (if PAT available) ────────────────────────────────
if PAT:
    import shutil
    models_dir = f"{REPO_DIR}/models"
    os.makedirs(models_dir, exist_ok=True)
    
    shutil.copy2(K12_MANIFOLD_PATH, f"{models_dir}/FLOW-K12_manifold.npz")
    shutil.copy2(K12_VOCAB_PATH,    f"{models_dir}/FLOW-K12_vocab.npz")
    
    cmds = [
        ["git", "add", "models/FLOW-K12_manifold.npz", "models/FLOW-K12_vocab.npz"],
        ["git", "commit", "-m", f"K-12 pipeline: {pipeline.n_concepts:,} concepts, {len(k12_entries):,} entries"],
        ["git", "push", "origin", "main"],
    ]
    for cmd in cmds:
        r = subprocess.run(cmd, cwd=REPO_DIR, capture_output=True, text=True)
        if r.returncode != 0 and 'nothing to commit' not in r.stdout:
            print(f"  \u2718 {' '.join(cmd[:3])}: {r.stderr.strip()[:100]}")
    print(f"\n\u2713 Pushed to GitHub")
else:
    print(f"\n\u26a0 No PAT \u2014 artifacts saved to /kaggle/working/ only")

## 11. Final Report

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#   FINAL REPORT — FLOW K-12 Progressive Pipeline
# ══════════════════════════════════════════════════════════════════════

print("\u2554" + "\u2550" * 68 + "\u2557")
print("\u2551" + "  FLOW K-12 Progressive Education Pipeline — Final Report".ljust(68) + "\u2551")
print("\u2560" + "\u2550" * 68 + "\u2563")

print("\u2551" + "  Architecture".ljust(68) + "\u2551")
print("\u2551" + f"    Manifold dimension  : {pipeline.manifold.DIM}".ljust(68) + "\u2551")
print("\u2551" + f"    Grammar engine      : {'ACTIVE' if grammar_active else 'INACTIVE'}".ljust(68) + "\u2551")
print("\u2551" + f"    Design              : Weight-free, Token-free, Training-free".ljust(68) + "\u2551")
print("\u2551" + "".ljust(68) + "\u2551")

print("\u2551" + "  Manifold State".ljust(68) + "\u2551")
print("\u2551" + f"    Total concepts      : {pipeline.n_concepts:,}".ljust(68) + "\u2551")
print("\u2551" + f"    Temperature         : {pipeline.temperature:.6f}".ljust(68) + "\u2551")
print("\u2551" + f"    Queries executed    : {pipeline.query_count}".ljust(68) + "\u2551")
print("\u2551" + "".ljust(68) + "\u2551")

print("\u2551" + "  K-12 Curriculum".ljust(68) + "\u2551")
print("\u2551" + f"    Grade levels        : K through 12 (13 levels)".ljust(68) + "\u2551")
print("\u2551" + f"    Vocabulary words    : {total_words}".ljust(68) + "\u2551")
print("\u2551" + f"    Curriculum concepts : {total_concepts}".ljust(68) + "\u2551")
print("\u2551" + f"    Contrast pairs      : {total_pairs}".ljust(68) + "\u2551")
print("\u2551" + f"    Expression entries  : {len(k12_entries):,}".ljust(68) + "\u2551")
print("\u2551" + "".ljust(68) + "\u2551")

print("\u2551" + "  Evaluation".ljust(68) + "\u2551")
print("\u2551" + f"    Mean coherence      : {suite.mean_coherence:.4f}".ljust(68) + "\u2551")
print("\u2551" + f"    Novelty decaying    : {suite.novelty_is_decaying}".ljust(68) + "\u2551")
print("\u2551" + "".ljust(68) + "\u2551")

print("\u2551" + "  Per-Grade Summary".ljust(68) + "\u2551")
for gk, gs in grade_stats.items():
    line = f"    {gs['label']:<12s}  +{gs['words_placed'] + gs['concepts_placed']:>3d} concepts  T={gs['temperature']:.4f}"
    print("\u2551" + line.ljust(68) + "\u2551")

print("\u2551" + "".ljust(68) + "\u2551")
print("\u2551" + "  Artifacts".ljust(68) + "\u2551")
print("\u2551" + f"    FLOW-K12_manifold.npz".ljust(68) + "\u2551")
print("\u2551" + f"    FLOW-K12_vocab.npz".ljust(68) + "\u2551")
print("\u2551" + "".ljust(68) + "\u2551")
print("\u2551" + "  To reload:".ljust(68) + "\u2551")
print("\u2551" + "    pipeline = GEOPipeline.load(".ljust(68) + "\u2551")
print("\u2551" + "        'FLOW-K12_manifold.npz',".ljust(68) + "\u2551")
print("\u2551" + "        vocabulary_path='FLOW-K12_vocab.npz')".ljust(68) + "\u2551")

print("\u255a" + "\u2550" * 68 + "\u255d")